In [14]:
import pandas as pd
import numpy as np

In [15]:
df = pd.read_csv('results.csv')
df.head()

,qnt,c1_p,c2_p,c3_p,c4_p,c5_p,c6_p,c1_p_normalized,c2_p_normalized,c3_p_normalized,c4_p_normalized,c5_p_normalized,c6_p_normalized,c1_real,c2_real,c3_real,c4_real,c5_real,c6_real
0,CC,0.91,0.67,0.75,0.90,0.92,0.84,0.18,0.13,0.15,0.18,0.18,0.17,0.09,0.34,0.25,0.07,0.08,0.17
1,PCC,0.91,0.68,0.75,0.88,0.92,0.84,0.18,0.14,0.15,0.18,0.18,0.17,0.09,0.34,0.25,0.07,0.08,0.17
2,ACC,0.09,0.26,0.25,0.00,0.06,0.13,0.11,0.33,0.32,0.00,0.08,0.16,0.09,0.34,0.25,0.07,0.08,0.17
3,PACC,0.09,0.25,0.25,0.00,0.06,0.13,0.12,0.31,0.32,0.00,0.08,0.17,0.09,0.34,0.25,0.07,0.08,0.17
4,T50,0.09,0.00,0.24,0.00,0.00,0.00,0.27,0.00,0.73,0.00,0.00,0.00,0.09,0.34,0.25,0.07,0.08,0.17


In [16]:
# Extract unique class names from column names
classes = sorted(set([col.split('_')[0][1:] for col in df.columns if col.startswith('c') and '_' in col]))
classes

['1', '2', '3', '4', '5', '6']

In [17]:
# Calculate absolute error metrics
for cls in classes:
    df[f'c{cls}_error'] = abs(df[f'c{cls}_real'] - df[f'c{cls}_p'])
    df[f'c{cls}_error_normalized'] = abs(df[f'c{cls}_real'] - df[f'c{cls}_p_normalized'])

In [18]:
df

,qnt,c1_p,c2_p,c3_p,c4_p,c5_p,c6_p,c1_p_normalized,c2_p_normalized,c3_p_normalized,...,c2_error,c2_error_normalized,c3_error,c3_error_normalized,c4_error,c4_error_normalized,c5_error,c5_error_normalized,c6_error,c6_error_normalized
0,CC,0.91,0.67,0.75,0.90,0.92,0.84,0.18,0.13,0.15,...,0.33,0.21,0.50,0.10,0.83,0.11,0.84,0.10,0.67,0.00
1,PCC,0.91,0.68,0.75,0.88,0.92,0.84,0.18,0.14,0.15,...,0.34,0.20,0.50,0.10,0.81,0.11,0.84,0.10,0.67,0.00
2,ACC,0.09,0.26,0.25,0.00,0.06,0.13,0.11,0.33,0.32,...,0.08,0.01,0.00,0.07,0.07,0.07,0.02,0.00,0.04,0.01
3,PACC,0.09,0.25,0.25,0.00,0.06,0.13,0.12,0.31,0.32,...,0.09,0.03,0.00,0.07,0.07,0.07,0.02,0.00,0.04,0.00
4,T50,0.09,0.00,0.24,0.00,0.00,0.00,0.27,0.00,0.73,...,0.34,0.34,0.01,0.48,0.07,0.07,0.08,0.08,0.17,0.17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21995,T50_syn,0.31,0.90,0.85,0.96,0.86,0.29,0.07,0.22,0.20,...,0.83,0.15,0.77,0.12,0.93,0.20,0.79,0.14,0.10,0.32
21996,MS_syn,0.64,0.93,0.92,0.98,0.93,0.61,0.13,0.19,0.18,...,0.86,0.12,0.84,0.10,0.95,0.17,0.86,0.12,0.22,0.27
21997,MS2_syn,0.64,0.93,0.92,0.98,0.93,0.61,0.13,0.19,0.18,...,0.86,0.12,0.84,0.10,0.95,0.17,0.86,0.12,0.22,0.27
21998,SMM_syn,0.67,1.00,1.00,1.00,1.00,0.66,0.13,0.19,0.19,...,0.93,0.12,0.92,0.11,0.97,0.16,0.93,0.12,0.27,0.27


## Analysis

### Normalized vs Standard Error

In [24]:
def plot_error_distribution(df):
    """
    Create an interactive box plot showing error distribution by class and quantification method.
    
    Parameters:
    df (pd.DataFrame): DataFrame containing error metrics and quantification methods
    
    Returns:
    plotly.graph_objects.Figure: Interactive plotly figure
    """
    import plotly.graph_objects as go
    
    # Get unique quantification methods
    qnt_methods = df['qnt'].unique().tolist()
    
    # Create figure
    fig = go.Figure()
    
    # Add traces for each qnt method
    for qnt in qnt_methods:
        df_filtered = df[df['qnt'] == qnt]
        
        # Add normalized and non-normalized traces for each class
        for i in classes:
            # Normalized error
            fig.add_trace(go.Box(
                y=df_filtered[f'c{i}_error_normalized'],
                name=f'Class {i} (Norm)',
                visible=(qnt == qnt_methods[0]),  # Only first method visible initially
                boxmean=True,
                marker=dict(color='lightblue'),
                legendgroup=f'class{i}',
                showlegend=True
            ))
            
            # Non-normalized error
            fig.add_trace(go.Box(
                y=df_filtered[f'c{i}_error'],
                name=f'Class {i}',
                visible=(qnt == qnt_methods[0]),  # Only first method visible initially
                boxmean=True,
                marker=dict(color='lightcoral'),
                legendgroup=f'class{i}',
                showlegend=True
            ))
    
    # Create buttons for dropdown
    buttons = []
    
    for idx, qnt in enumerate(qnt_methods):
        # Calculate which traces should be visible for this qnt method
        visible = [False] * len(fig.data)
        start_idx = idx * 12  # 12 traces per qnt method (6 classes × 2 error types)
        for i in range(12):
            visible[start_idx + i] = True
        
        buttons.append(dict(
            label=qnt,
            method='update',
            args=[{'visible': visible}]
        ))
    
    # Update layout
    fig.update_layout(
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            x=0.17,
            xanchor='left',
            y=1.15,
            yanchor='top'
        )],
        title='Error Distribution by Class and Quantification Method',
        xaxis_title='Class and Error Type',
        yaxis_title='Error',
        height=600,
        showlegend=True
    )
    
    return fig

# Call the function and display the plot
fig = plot_error_distribution(df)
fig.show()


### Boxplot per method

In [20]:
def plot_normalized_error_by_method(df):
    """
    Create an interactive box plot showing normalized error distribution by class for each quantification method.
    
    Parameters:
    df (pd.DataFrame): DataFrame containing error metrics and quantification methods
    
    Returns:
    plotly.graph_objects.Figure: Interactive plotly figure
    """
    import plotly.graph_objects as go
    
    # Create figure
    fig = go.Figure()
    
    # Get unique quantification methods
    qnt_methods = df['qnt'].unique().tolist()
    
    # Add traces for each qnt method
    for qnt in qnt_methods:
        df_filtered = df[df['qnt'] == qnt]
        
        # Add trace for each class
        for i in classes:
            fig.add_trace(go.Box(
                y=df_filtered[f'c{i}_error_normalized'],
                name=f'Class {i}',
                visible=(qnt == qnt_methods[0]),  # Only first method visible initially
                boxmean=True
            ))
    
    # Create buttons for dropdown
    buttons = []
    for idx, qnt in enumerate(qnt_methods):
        # Calculate which traces should be visible for this qnt method
        visible = [False] * len(fig.data)
        start_idx = idx * 6  # 6 traces per qnt method (one per class)
        for i in range(6):
            visible[start_idx + i] = True
        
        buttons.append(dict(
            label=qnt,
            method='update',
            args=[{'visible': visible}]
        ))
    
    # Update layout
    fig.update_layout(
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            x=0.17,
            xanchor='left',
            y=1.15,
            yanchor='top'
        )],
        title='Normalized Error Distribution by Class',
        xaxis_title='Class',
        yaxis_title='Normalized Error',
        height=600,
        showlegend=True
    )
    
    return fig

# Call the function and display the plot
fig2 = plot_normalized_error_by_method(df)
fig2.show()

In [21]:
def plot_traditional_vs_syn_comparison(df):
    """
    Create an interactive box plot comparing traditional quantification methods with their synthetic counterparts.
    
    Parameters:
    df (pd.DataFrame): DataFrame containing error metrics and quantification methods
    
    Returns:
    plotly.graph_objects.Figure: Interactive plotly figure
    """
    import plotly.graph_objects as go
    
    # Define method pairs (traditional, synthetic)
    method_pairs = {
        'ACC': ('ACC', 'ACC_syn'),
        'PACC': ('PACC', 'PACC_syn'),
        'X': ('X', 'X_syn'),
        'MAX': ('MAX', 'MAX_syn'),
        'T50': ('T50', 'T50_syn'),
        'MS': ('MS', 'MS_syn'),
        'MS2': ('MS2', 'MS2_syn'),
        'SMM': ('SMM', 'SMM_syn'),
        'HDy': ('HDy', 'HDy_syn'),
        'DyS': ('DyS', 'DySyn')
    }
    
    # Create figure
    fig = go.Figure()
    
    # Add traces for each method pair
    for method_name, (trad_method, syn_method) in method_pairs.items():
        # Filter data for traditional method
        df_trad = df[df['qnt'] == trad_method]
        df_syn = df[df['qnt'] == syn_method]
        
        # Add traces for each class
        for i in classes:
            # Traditional method
            if len(df_trad) > 0:
                fig.add_trace(go.Box(
                    y=df_trad[f'c{i}_error_normalized'],
                    name=f'Class {i} (Trad)',
                    visible=(method_name == 'ACC'),
                    boxmean=True,
                    marker=dict(color='lightcoral'),
                    legendgroup=f'class{i}',
                    showlegend=True,
                    offsetgroup=f'c{i}_trad',
                    width=0.3
                ))
            
            # Synthetic method
            if len(df_syn) > 0:
                fig.add_trace(go.Box(
                    y=df_syn[f'c{i}_error_normalized'],
                    name=f'Class {i} (Syn)',
                    visible=(method_name == 'ACC'),
                    boxmean=True,
                    marker=dict(color='lightblue'),
                    legendgroup=f'class{i}',
                    showlegend=True,
                    offsetgroup=f'c{i}_syn',
                    width=0.3
                ))
    
    # Create buttons for dropdown
    buttons = []
    for idx, method_name in enumerate(method_pairs.keys()):
        # Calculate which traces should be visible
        visible = [False] * len(fig.data)
        start_idx = idx * 12  # 12 traces per method pair (6 classes × 2 types)
        for i in range(12):
            visible[start_idx + i] = True
        
        buttons.append(dict(
            label=method_name,
            method='update',
            args=[{'visible': visible}]
        ))
    
    # Update layout
    fig.update_layout(
        updatemenus=[dict(
            active=0,
            buttons=buttons,
            x=0.17,
            xanchor='left',
            y=1.15,
            yanchor='top'
        )],
        title='Traditional vs Synthetic Methods: Normalized Error by Class',
        xaxis_title='Class and Method Type',
        yaxis_title='Normalized Error',
        height=600,
        showlegend=True,
        boxmode='group'
    )
    
    return fig

# Call the function and display the plot
fig3 = plot_traditional_vs_syn_comparison(df)
fig3.show()


In [22]:
df[df['qnt'] == 'DyS'].columns

Index(['qnt', 'c1_p', 'c2_p', 'c3_p', 'c4_p', 'c5_p', 'c6_p',
       'c1_p_normalized', 'c2_p_normalized', 'c3_p_normalized',
       'c4_p_normalized', 'c5_p_normalized', 'c6_p_normalized', 'c1_real',
       'c2_real', 'c3_real', 'c4_real', 'c5_real', 'c6_real', 'c1_error',
       'c1_error_normalized', 'c2_error', 'c2_error_normalized', 'c3_error',
       'c3_error_normalized', 'c4_error', 'c4_error_normalized', 'c5_error',
       'c5_error_normalized', 'c6_error', 'c6_error_normalized'],
      dtype='object')

### Win Plot

In [23]:
def plot_win_comparison(df):
    """
    Create a horizontal bar plot comparing wins between traditional and synthetic methods.
    A method wins on a class if it has lower mean error than its counterpart.
    
    Parameters:
    df (pd.DataFrame): DataFrame containing error metrics and quantification methods
    
    Returns:
    plotly.graph_objects.Figure: Interactive plotly figure
    """
    import plotly.graph_objects as go
    
    # Define method pairs (traditional, synthetic)
    method_pairs = {
        'ACC': ('ACC', 'ACC_syn'),
        'PACC': ('PACC', 'PACC_syn'),
        'X': ('X', 'X_syn'),
        'MAX': ('MAX', 'MAX_syn'),
        'T50': ('T50', 'T50_syn'),
        'MS': ('MS', 'MS_syn'),
        'MS2': ('MS2', 'MS2_syn'),
        'SMM': ('SMM', 'SMM_syn'),
        'HDy': ('HDy', 'HDy_syn'),
        'DyS': ('DyS', 'DySyn')
    }
    
    # Calculate wins for each method
    results = []
    
    for method_name, (trad_method, syn_method) in method_pairs.items():
        trad_wins = 0
        syn_wins = 0
        
        # Check each class
        for i in classes:
            df_trad = df[df['qnt'] == trad_method]
            df_syn = df[df['qnt'] == syn_method]
            
            if len(df_trad) > 0 and len(df_syn) > 0:
                trad_mean = df_trad[f'c{i}_error_normalized'].mean()
                syn_mean = df_syn[f'c{i}_error_normalized'].mean()
                
                if trad_mean < syn_mean:
                    trad_wins += 1
                elif syn_mean < trad_mean:
                    syn_wins += 1
        
        results.append({
            'method': method_name,
            'trad_wins': trad_wins,
            'syn_wins': syn_wins
        })
    
    # Create figure
    fig = go.Figure()
    
    # Extract data
    methods = [r['method'] for r in results]
    trad_wins = [-r['trad_wins'] for r in results]  # Negative for left side
    syn_wins = [r['syn_wins'] for r in results]
    
    # Add traditional wins (left side, negative values)
    fig.add_trace(go.Bar(
        y=methods,
        x=trad_wins,
        name='Traditional',
        orientation='h',
        marker=dict(color='lightcoral'),
        text=[-x for x in trad_wins],
        textposition='auto',
    ))
    
    # Add synthetic wins (right side, positive values)
    fig.add_trace(go.Bar(
        y=methods,
        x=syn_wins,
        name='Synthetic',
        orientation='h',
        marker=dict(color='lightblue'),
        text=syn_wins,
        textposition='auto',
    ))
    
    # Update layout
    fig.update_layout(
        title='Traditional vs Synthetic: Win Count by Method (6 classes)',
        xaxis_title='Number of Wins',
        yaxis_title='Method',
        barmode='relative',
        height=500,
        xaxis=dict(
            tickvals=[-6, -4, -2, 0, 2, 4, 6],
            ticktext=['6', '4', '2', '0', '2', '4', '6']
        ),
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    return fig

# Call the function and display the plot
fig_wins = plot_win_comparison(df)
fig_wins.show()